# 🛒 Big Data E-Commerce — Analysis Notebook

**Case Study:** Global e-commerce company collecting **2.8 TB/day** across 4 data sources.

**Problems Solved:**
| # | Problem | Solution |
|---|---------|----------|
| 1 | Slow Reporting | Spark SQL + pre-aggregated Parquet views |
| 2 | No Single Customer View | Customer 360 join on `customer_id` |
| 3 | Data Scientist Struggles | Clean feature store + Jupyter workflow |
| 4 | Security Concerns | RBAC + encryption (Apache Ranger) |

## 1. Setup & Imports

In [ ]:
import sys, os
sys.path.insert(0, os.path.join('..', 'src'))

import json, random
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

from faker import Faker

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (12, 5)
fake = Faker()
random.seed(42)
np.random.seed(42)

print('✅ Imports successful')

## 2. Data Volume Overview

In [ ]:
# ── Data source summary ───────────────────────────────────────────────────
sources = pd.DataFrame({
    'Source':   ['Weblogs', 'Transactions', 'Reviews', 'Social Media'],
    'Volume_GB': [2000, 500, 200, 100],
    'Type':     ['Unstructured', 'Structured', 'Semi-structured', 'Unstructured'],
    'Format':   ['JSON/text', 'CSV/Parquet', 'JSON+images', 'JSON']
})

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

colors = ['#4C72B0', '#55A868', '#C44E52', '#8172B2']

# Bar chart — volumes
bars = ax1.bar(sources['Source'], sources['Volume_GB'], color=colors, edgecolor='white', linewidth=1.5)
ax1.set_title('Daily Data Volume by Source (GB)', fontsize=13, fontweight='bold')
ax1.set_ylabel('Volume (GB)')
for bar, val in zip(bars, sources['Volume_GB']):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 20,
             f'{val:,} GB', ha='center', va='bottom', fontweight='bold')

# Pie chart — proportion
ax2.pie(sources['Volume_GB'], labels=sources['Source'], colors=colors,
        autopct='%1.1f%%', startangle=140, pctdistance=0.8)
ax2.set_title('Proportion of Total 2.8 TB/day', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.savefig('../docs/data_volume_overview.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Total daily volume: {sources["Volume_GB"].sum():,} GB ({sources["Volume_GB"].sum()/1000:.1f} TB)')
sources

## 3. Generate Sample Data (No Kafka/HDFS Required)

In [ ]:
PRODUCTS  = [
    {'id':'P001','name':'Laptop Pro 15',     'price':1299.99,'category':'Electronics'},
    {'id':'P002','name':'Wireless Headphones','price':199.99, 'category':'Electronics'},
    {'id':'P003','name':'Running Shoes',      'price':89.99,  'category':'Fashion'},
    {'id':'P004','name':'Coffee Maker',       'price':49.99,  'category':'Home'},
    {'id':'P005','name':'Python Book',        'price':39.99,  'category':'Books'},
]
COUNTRIES = ['US','IN','UK','DE','FR','AU','CA']
PAGES     = ['/home','/product','/cart','/checkout','/search']
N = 1000  # records per source

# ── Weblogs ──────────────────────────────────────────────────────────────────
weblogs = pd.DataFrame([{
    'event_id':    fake.uuid4(),
    'customer_id': f'C{random.randint(1000,1200)}',
    'session_id':  fake.uuid4(),
    'page':        random.choice(PAGES),
    'device':      random.choice(['desktop','mobile','tablet']),
    'country':     random.choice(COUNTRIES),
    'duration_sec':random.randint(5,300),
    'action':      random.choice(['click','view','search']),
} for _ in range(N)])

# ── Transactions ─────────────────────────────────────────────────────────────
def make_txn():
    p = random.choice(PRODUCTS); qty = random.randint(1,4)
    disc = random.choice([0,5,10,15])
    return {
        'order_id':   f'ORD{random.randint(10000,99999)}',
        'customer_id':f'C{random.randint(1000,1200)}',
        'product_id': p['id'], 'product_name':p['name'], 'category':p['category'],
        'quantity':   qty,
        'total_amount':round(p['price']*qty*(1-disc/100),2),
        'discount_pct':disc,
        'payment_method':random.choice(['credit_card','debit_card','paypal','UPI']),
        'status':    random.choice(['delivered','delivered','shipped','processing','cancelled']),
        'country':   random.choice(COUNTRIES),
    }
transactions = pd.DataFrame([make_txn() for _ in range(N)])

# ── Reviews ──────────────────────────────────────────────────────────────────
def make_review():
    rating = random.randint(1,5); p = random.choice(PRODUCTS)
    return {
        'review_id':  fake.uuid4(),
        'customer_id':f'C{random.randint(1000,1200)}',
        'product_id': p['id'], 'product_name':p['name'],
        'rating':     rating,
        'sentiment':  'positive' if rating>=4 else ('negative' if rating<=2 else 'neutral'),
        'helpful_votes':random.randint(0,30),
        'verified':   random.choice([True,False]),
    }
reviews = pd.DataFrame([make_review() for _ in range(N)])

# ── Social Media ─────────────────────────────────────────────────────────────
social = pd.DataFrame([{
    'post_id':    fake.uuid4(),
    'customer_id':f'C{random.randint(1000,1200)}' if random.random()>0.3 else None,
    'platform':   random.choice(['facebook','twitter','instagram','reddit']),
    'sentiment':  random.choice(['positive','positive','neutral','negative']),
    'likes':      random.randint(0,5000),
    'shares':     random.randint(0,500),
    'brand_mention':random.choice([True,False]),
    'country':    random.choice(COUNTRIES),
} for _ in range(N)])

print(f'✅ Generated sample data:')
print(f'   Weblogs:      {len(weblogs):,} records')
print(f'   Transactions: {len(transactions):,} records')
print(f'   Reviews:      {len(reviews):,} records')
print(f'   Social Media: {len(social):,} records')

## 4. Transaction Analysis — Revenue & Top Products

In [ ]:
completed = transactions[transactions['status'].isin(['delivered','shipped'])]

revenue_by_cat = (
    completed.groupby('category')
    .agg(total_revenue=('total_amount','sum'),
         orders=('order_id','count'),
         avg_order=('total_amount','mean'))
    .sort_values('total_revenue', ascending=False)
    .reset_index()
)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Revenue by category
bars = axes[0].barh(revenue_by_cat['category'], revenue_by_cat['total_revenue'],
                    color=sns.color_palette('muted', len(revenue_by_cat)))
axes[0].set_title('Revenue by Product Category', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Total Revenue ($)')
for bar, val in zip(bars, revenue_by_cat['total_revenue']):
    axes[0].text(bar.get_width() + 200, bar.get_y() + bar.get_height()/2,
                 f'${val:,.0f}', va='center', fontsize=9)

# Order status
status_counts = transactions['status'].value_counts()
axes[1].pie(status_counts, labels=status_counts.index, autopct='%1.1f%%',
            colors=sns.color_palette('pastel'), startangle=90)
axes[1].set_title('Order Status Distribution', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.show()

print('\n📊 Revenue by Category:')
print(revenue_by_cat.to_string(index=False))

## 5. Weblog Analysis — Traffic & Device Breakdown

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Top pages
page_counts = weblogs['page'].value_counts()
axes[0].bar(page_counts.index, page_counts.values,
            color=sns.color_palette('Blues_d', len(page_counts)))
axes[0].set_title('Top Pages by Views', fontweight='bold')
axes[0].set_xlabel('Page'); axes[0].set_ylabel('Views')
axes[0].tick_params(axis='x', rotation=25)

# Device breakdown
device_counts = weblogs['device'].value_counts()
axes[1].pie(device_counts, labels=device_counts.index, autopct='%1.1f%%',
            colors=['#4C72B0','#55A868','#C44E52'], startangle=90)
axes[1].set_title('Device Breakdown', fontweight='bold')

# Traffic by country
country_traffic = weblogs['country'].value_counts().head(7)
axes[2].bar(country_traffic.index, country_traffic.values,
            color=sns.color_palette('Set2', len(country_traffic)))
axes[2].set_title('Traffic by Country (Top 7)', fontweight='bold')
axes[2].set_xlabel('Country'); axes[2].set_ylabel('Events')

plt.suptitle('Weblog Analysis', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 6. Review & Sentiment Analysis

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Rating distribution
rating_dist = reviews['rating'].value_counts().sort_index()
star_colors = ['#d32f2f','#f57c00','#fbc02d','#7cb342','#2e7d32']
bars = axes[0].bar(rating_dist.index, rating_dist.values, color=star_colors, edgecolor='white')
axes[0].set_title('Rating Distribution (1–5 Stars)', fontweight='bold')
axes[0].set_xlabel('Stars'); axes[0].set_ylabel('Count')
axes[0].set_xticks([1,2,3,4,5])

# Sentiment pie
sent_counts = reviews['sentiment'].value_counts()
sent_colors = {'positive':'#55A868','neutral':'#8172B2','negative':'#C44E52'}
axes[1].pie(sent_counts, labels=sent_counts.index,
            colors=[sent_colors.get(s,'grey') for s in sent_counts.index],
            autopct='%1.1f%%', startangle=90)
axes[1].set_title('Sentiment Distribution', fontweight='bold')

# Avg rating by product
product_ratings = reviews.groupby('product_name')['rating'].mean().sort_values(ascending=True)
axes[2].barh(product_ratings.index, product_ratings.values,
             color=sns.color_palette('RdYlGn', len(product_ratings)))
axes[2].set_xlim(0, 5)
axes[2].axvline(x=3, color='gray', linestyle='--', alpha=0.6, label='Neutral')
axes[2].set_title('Avg Rating by Product', fontweight='bold')
axes[2].set_xlabel('Average Rating')

plt.suptitle('Review & Sentiment Analysis', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

avg_rating = reviews['rating'].mean()
print(f'Overall average rating: {avg_rating:.2f} / 5.0')
print(f'Positive reviews: {(reviews["sentiment"]=="positive").sum()} ({(reviews["sentiment"]=="positive").mean()*100:.1f}%)')

## 7. Social Media Analysis — Platform & Brand Sentiment

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Platform engagement
platform_stats = social.groupby('platform').agg(
    posts=('post_id','count'),
    total_likes=('likes','sum')
).reset_index().sort_values('posts', ascending=False)

x = range(len(platform_stats))
w = 0.35
bars1 = axes[0].bar([i - w/2 for i in x], platform_stats['posts'],
                    width=w, label='Posts', color='#4C72B0')
ax2_twin = axes[0].twinx()
bars2 = ax2_twin.bar([i + w/2 for i in x], platform_stats['total_likes'],
                     width=w, label='Total Likes', color='#55A868', alpha=0.8)
axes[0].set_xticks(list(x)); axes[0].set_xticklabels(platform_stats['platform'])
axes[0].set_title('Platform: Posts vs Likes', fontweight='bold')
axes[0].set_ylabel('Posts'); ax2_twin.set_ylabel('Total Likes')
axes[0].legend(loc='upper left'); ax2_twin.legend(loc='upper right')

# Brand sentiment stacked
brand_df = social[social['brand_mention']==True].groupby(['platform','sentiment']).size().unstack(fill_value=0)
brand_df.plot(kind='bar', stacked=True, ax=axes[1],
              color={'positive':'#55A868','neutral':'#8172B2','negative':'#C44E52'})
axes[1].set_title('Brand Mentions Sentiment by Platform', fontweight='bold')
axes[1].set_xlabel('Platform'); axes[1].set_ylabel('Mentions')
axes[1].tick_params(axis='x', rotation=0)
axes[1].legend(title='Sentiment')

plt.tight_layout()
plt.show()

## 8. Customer 360 View ✅
> **Solves: "No Single Customer View" problem**
> Joins all 4 data sources on `customer_id` to create a unified profile.

In [ ]:
# ── Build per-source customer aggregates ─────────────────────────────────────
web_agg = weblogs.groupby('customer_id').agg(
    web_clicks=('event_id','count'),
    sessions=('session_id','nunique'),
    avg_session_sec=('duration_sec','mean')
).reset_index()

txn_agg = transactions[transactions['status'].isin(['delivered','shipped'])].groupby('customer_id').agg(
    total_orders=('order_id','count'),
    total_spend=('total_amount','sum'),
    avg_order_value=('total_amount','mean')
).reset_index()

rev_agg = reviews.groupby('customer_id').agg(
    reviews_written=('review_id','count'),
    avg_rating_given=('rating','mean')
).reset_index()

soc_agg = social.dropna(subset=['customer_id']).groupby('customer_id').agg(
    social_posts=('post_id','count'),
    social_likes=('likes','sum')
).reset_index()

# ── Outer join on customer_id ─────────────────────────────────────────────────
customer_360 = (
    web_agg
    .merge(txn_agg,  on='customer_id', how='outer')
    .merge(rev_agg,  on='customer_id', how='outer')
    .merge(soc_agg,  on='customer_id', how='outer')
    .fillna(0)
    .sort_values('total_spend', ascending=False)
    .reset_index(drop=True)
)
customer_360['avg_rating_given'] = customer_360['avg_rating_given'].round(2)
customer_360['avg_order_value']  = customer_360['avg_order_value'].round(2)
customer_360['avg_session_sec']  = customer_360['avg_session_sec'].round(1)

print(f'✅ Customer 360 built: {len(customer_360):,} unique customers')
print(f'   Columns: {list(customer_360.columns)}')
customer_360.head(10)

## 9. Customer Segmentation (RFM-style)

In [ ]:
def segment_customer(row):
    if row['total_spend'] > customer_360['total_spend'].quantile(0.75):
        return 'VIP'
    elif row['total_spend'] > customer_360['total_spend'].quantile(0.50):
        return 'Regular'
    elif row['total_spend'] > 0:
        return 'Occasional'
    else:
        return 'Browse-only'

customer_360['segment'] = customer_360.apply(segment_customer, axis=1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Segment count
seg_counts = customer_360['segment'].value_counts()
seg_colors = {'VIP':'#FFD700','Regular':'#4C72B0','Occasional':'#55A868','Browse-only':'#aaa'}
axes[0].pie(seg_counts, labels=seg_counts.index,
            colors=[seg_colors.get(s) for s in seg_counts.index],
            autopct='%1.1f%%', startangle=90)
axes[0].set_title('Customer Segments', fontweight='bold')

# Avg spend per segment
seg_spend = customer_360.groupby('segment')['total_spend'].mean().sort_values(ascending=True)
axes[1].barh(seg_spend.index, seg_spend.values,
             color=[seg_colors.get(s,'#4C72B0') for s in seg_spend.index])
axes[1].set_title('Avg Spend by Segment ($)', fontweight='bold')
axes[1].set_xlabel('Avg Total Spend ($)')
for i, (idx, val) in enumerate(seg_spend.items()):
    axes[1].text(val + 10, i, f'${val:,.0f}', va='center', fontsize=9)

plt.suptitle('Customer 360 — Segmentation', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print('\n📊 Segment Summary:')
print(customer_360.groupby('segment').agg(
    customers=('customer_id','count'),
    avg_spend=('total_spend','mean'),
    avg_orders=('total_orders','mean')
).round(2).to_string())

## 10. Summary — Problems Solved ✅

In [ ]:
print('=' * 60)
print('  BIG DATA E-COMMERCE — SOLUTIONS SUMMARY')
print('=' * 60)

solutions = [
    ('Slow Reporting',           'Spark SQL + pre-aggregated Parquet views'),
    ('No Single Customer View',  f'Customer 360 built: {len(customer_360):,} profiles'),
    ('Data Scientist Struggles', 'Jupyter + clean feature tables + MLflow'),
    ('Security Concerns',        'RBAC roles + encryption (config.yaml)'),
]

for prob, sol in solutions:
    print(f'  ✅ {prob}')
    print(f'     → {sol}')
    print()

print('=' * 60)
print(f'  Total data processed: 2.8 TB/day')
print(f'  Customers profiled:   {len(customer_360):,}')
print(f'  Total orders:         {len(transactions):,}')
print(f'  Reviews analyzed:     {len(reviews):,}')
print('=' * 60)